# ️ Glu-Stock: 04_MONITOR_ALERT
**Phase**: Real-time Monitoring & Telegram Reporting

In [ ]:
!pip install -q pyTelegramBotAPI firebase-admin python-dotenv


In [ ]:
# [INFO] SECTION 2: INFRASTRUCTURE (Firebase, Secrets & Universe)
import json, os, firebase_admin, joblib, numpy as np, pandas as pd, yfinance as yf, warnings
from firebase_admin import credentials, firestore
from datetime import datetime
warnings.filterwarnings('ignore')

try:
    from kaggle_secrets import UserSecretsClient
    IS_KAGGLE = True
except ImportError:
    IS_KAGGLE = False

class KaggleInfra:
    @staticmethod
    def load_secrets():
        if IS_KAGGLE:
            user_secrets = UserSecretsClient()
            try:
                raw = user_secrets.get_secret("FIREBASE_KEY_JSON")
                return {"key": json.loads(raw)}
            except Exception as e:
                print(f"[ERROR] FIREBASE_KEY_JSON missing or invalid! Error: {e}")
                return {"key": None}
        else:
            from dotenv import load_dotenv
            load_dotenv()
            raw = os.getenv("FIREBASE_KEY_JSON")
            if not raw: return {"key": None}
            return {"key": json.loads(raw)}

class FirebaseHandler:
    def __init__(self, secrets):
        if not firebase_admin._apps:
            if not secrets.get('key'):
                raise ValueError("FIREBASE_KEY_JSON is missing. Check Kaggle Secrets.")
            cred = credentials.Certificate(secrets['key'])
            firebase_admin.initialize_app(cred)
        self.db = firestore.client()
    def get_and_clear_queue(self, queue_name: str):
        docs = self.db.collection(f"glu_stock_queue_{queue_name}").get()
        tasks = []
        for doc in docs:
            dt = doc.to_dict()
            tasks.append(dt.get('payload', dt))
            doc.reference.delete()
        return tasks
    def push_task(self, queue_name: str, data):
        self.db.collection(f"glu_stock_queue_{queue_name}").add({'payload': data, 'timestamp': datetime.now().isoformat()})
    def log_event(self, phase, details):
        self.db.collection("glu_stock_history").add({'timestamp': datetime.now().isoformat(), 'phase': phase.upper(), 'details': details})
    def wait_for_queue(self, queue_name: str, max_retries=20, interval=60):
        import time
        for i in range(max_retries):
            docs = self.db.collection(f"glu_stock_queue_{queue_name}").get()
            if docs:
                return self.get_and_clear_queue(queue_name)
            if i < max_retries - 1:
                print(f"[WAIT] {queue_name} queue empty. Retrying ({i+1}/{max_retries}) in {interval}s...", flush=True)
                time.sleep(interval)
        return []


In [ ]:
def run_monitor():
    secrets = KaggleInfra.load_secrets()
    fb = FirebaseHandler(secrets)
    bot = telebot.TeleBot(secrets['telegram'])
    chat_id = "INSERT_YOUR_CHAT_ID_HERE"
    history = fb.get_history()
    active = fb.get_active_trades()
    msg = f"️ **CLOUD STATUS** ️\n **Active**: {len(active)}\n\n **Latest**:\n"
    if history:
        for k, v in history.items(): msg += f"- [{v['phase']}] {v['details'][:40]}...\n"
    try: bot.send_message(chat_id, msg, parse_mode="Markdown")
    except: print(msg)
run_monitor()